In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# EXAMPLE = 'apt0_body_state'
# IMG_NAME = 'kick-crop.jpeg'
# IMG_NAME = 'kick.jpeg'
# IMG_NAME = 'at-chair.jpeg'
# IMG_NAME = 'mid-air.jpg'
# IMG_NAME = 'pre-jump.jpg'

EXAMPLE = 'groceries_world'
IMG_NAME = 'start.png'
# IMG_NAME = 'holding_milk.png'
# IMG_NAME = 'holding_cereal.png'
# IMG_NAME = 'cereal_in_container.png'
# IMG_NAME = 'milk_in_container.png'

# LLAMA_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
LLAMA_ID = "meta-llama/Meta-Llama-3-70B-Instruct"
# LLAMA_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# LLAVA_ID = "liuhaotian/llava-v1.6-34b"
# LLAVA_ID = "lmms-lab/llava-onevision-qwen2-0.5b-si"
LLAVA_ID = "lmms-lab/llava-onevision-qwen2-7b-si"
# LLAVA_ID = "lmms-lab/llava-onevision-qwen2-7b-ov"

import warnings
warnings.filterwarnings("ignore")

In [3]:
from semantic_state_estimator import SemanticStateEstimator

se = SemanticStateEstimator(
    domain=f'examples/{EXAMPLE}/domain.pddl',
    problem=f'examples/{EXAMPLE}/problem.pddl',
    nl_converter_model_id=LLAMA_ID,
    vqa_model_id=LLAVA_ID
)
se.queries_dict

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


predicate queries loaded from cache
Loaded LLaVA model: lmms-lab/llava-onevision-qwen2-7b-ov


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
You are using a model of type llava to instantiate a model of type llava_qwen. This is not supported for all configurations of models and can yield errors.


Loading vision tower: google/siglip-so400m-patch14-384


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model Class: LlavaQwenForCausalLM


{'on-table(milk-carton)': 'Is the milk carton on the table?',
 'on-table(lemon)': 'Is the lemon on the table?',
 'on-table(green-bottle)': 'Is the green bottle on the table?',
 'on-table(loaf-of-bread)': 'Is the loaf of bread on the table?',
 'on-table(red-box-of-cereal)': 'Is the red box of cereal on the table?',
 'on-table(red-can-of-soda)': 'Is the red can of soda on the table?',
 'gripper-empty()': 'Is the gripper currently empty?',
 'gripping(milk-carton)': 'Is the robot currently holding the milk carton?',
 'gripping(lemon)': 'Is the robot currently holding the lemon?',
 'gripping(green-bottle)': 'Is the robot currently holding the green bottle?',
 'gripping(loaf-of-bread)': 'Is the robot currently holding the loaf of bread?',
 'gripping(red-box-of-cereal)': 'Is the robot currently holding the red box of cereal?',
 'gripping(red-can-of-soda)': 'Is the robot currently gripping the red can of soda?',
 'in-container(milk-carton,dark-wood-container)': 'Is the milk carton in the dark 

In [4]:
from PIL import Image
import numpy as np


img = Image.open(f'examples/{EXAMPLE}/{IMG_NAME}')
img2 = Image.open(f'examples/apt0_body_state/kick-crop.jpeg')
arr = np.array(img)

In [ ]:
prob_map = se.estimate_state([img])
prob_map

  0%|          | 0/25 [00:00<?, ?it/s]

In [ ]:
state = set()
for pred, prob in prob_map.items():
    if prob > 0.5:
        state.add(pred)

state

In [7]:
se.vqa_model([img2, img], "describe the images")

"The first image shows a 3D-rendered scene of a room with a person in the center. The person appears to be wearing headphones and is in a dynamic pose, suggesting movement or dancing. They are dressed in yellow pants with blue stripes on the sides, white shoes with colorful accents, and a gray top. The room has a modern design with wooden flooring, a kitchen area in the background featuring cabinets and a countertop, and a dining area with chairs and a table. There's also a couch with patterned cushions visible.\n\nThe second image depicts a different scene with a focus on a robotic arm situated in the middle of a checkered floor. The robot has a cylindrical body with a jointed arm that ends in what looks like a gripper or tool. There are several objects placed around the robot: a small box, a bottle, and some other items that are not clearly identifiable. The setting seems to be an indoor space with a geometrically patterned floor, possibly for testing or demonstrating the robot's cap

In [ ]:
img2

In [ ]:
len(prob_map)

In [ ]:
se.queries_dict

In [ ]:
se.vqa_model('examples/apt0_body_state/right-foot-forward.jpeg', "Is the girl's left leg in front of her right leg?")

In [ ]:
probs = se.vqa_model('examples/apt0_body_state/right-foot-forward.jpeg', "Is the girl's left leg in front of her right leg?", get_probs=True)[-1]

In [ ]:
probs = se.vqa_model('examples/apt0_body_state/pre-jump.jpg', "Is the girl's left leg in front of her left leg?", get_probs=True)[-1]

In [ ]:
yes = probs[se.vqa_model.tokenizer.encode(['YES', 'yes', 'Yes'])].sum()
no = probs[se.vqa_model.tokenizer.encode(['NO', 'no', 'No'])].sum()

yes_prob = yes / (yes + no)
yes_prob.item()